# MatchFormer Epipolar Fine-tuning — Google Colab

Fine-tunes MatchFormer Lite-LA with epipolar supervision on ScanNet.
Checkpoints are saved to **Google Drive** so training survives Colab disconnects.

**Run order (fresh session):** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7  
**Resume after disconnect:** Cell 1 → 3 → 4 → 7 (skip 2, 5, 6)

## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ZIPS    = '/content/drive/MyDrive/final_proj/scannet_zips'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/final_proj/cr13'

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Zips dir:       {DRIVE_ZIPS}')
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

Mounted at /content/drive
Zips dir:       /content/drive/MyDrive/final_proj/scannet_zips
Checkpoint dir: /content/drive/MyDrive/final_proj/cr13


## Cell 2 — Clone Repo & Install Dependencies
*(Skip on resume — repo and packages persist for the session)*

In [2]:
REPO_URL = 'https://github.com/sid-raj-uc/match-former.git'
REPO_DIR = '/content/final_proj'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    !git clone -b develop {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_DIR} && git fetch origin && git checkout -f develop && git pull origin develop

%cd /content/final_proj/MatchFormer



Cloning repo...
Cloning into '/content/final_proj'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 345 (delta 101), reused 91 (delta 47), pack-reused 192 (from 1)
Receiving objects: 100% (345/345), 133.22 MiB | 46.91 MiB/s, done.
Resolving deltas: 100% (184/184), done.
/content/final_proj/MatchFormer


In [3]:
!pip install -q pytorch-lightning einops kornia timm loguru yacs
!pip install -q opencv-python-headless
print('Done.')

## Cell 3 — Verify Zips on Drive

In [4]:
import glob, os

%cd /content/final_proj/MatchFormer

zips = sorted(glob.glob(os.path.join(DRIVE_ZIPS, '*.zip')))
print(f'Found {len(zips)} zip(s) on Drive:')
for z in zips:
    size_gb = os.path.getsize(z) / 1e9
    print(f'  {os.path.basename(z)}  ({size_gb:.2f} GB)')

## Cell 4 — Extract Zips to Local SSD

Copies zips from Drive to Colab's local SSD and extracts them.
Drive has high per-file latency — local SSD is much faster for the dataloader.
Takes ~5–10 min once per session; already-extracted scenes are skipped.

In [ ]:
import os, glob, shutil, subprocess, time

LOCAL_DATA = '/content/scannet_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

zips = sorted(glob.glob(os.path.join(DRIVE_ZIPS, '*.zip')))
print(f'Found {len(zips)} zip(s) to extract\n')

failed = []
for i, zip_drive in enumerate(zips, 1):
    scene = os.path.basename(zip_drive).replace('.zip', '')
    dst = os.path.join(LOCAL_DATA, scene)
    prefix = f'[{i}/{len(zips)}] {scene}'

    if os.path.isdir(dst) and len(glob.glob(os.path.join(dst, 'exported/color/*.jpg'))) > 0:
        n = len(glob.glob(os.path.join(dst, 'exported/color/*.jpg')))
        print(f'{prefix} — already extracted ({n} frames), skipping')
        continue

    size_gb = os.path.getsize(zip_drive) / 1e9
    zip_local = f'/content/{scene}.zip'

    t0 = time.time()
    print(f'{prefix} — copying {size_gb:.2f} GB from Drive...', end=' ', flush=True)
    shutil.copy2(zip_drive, zip_local)
    print(f'done ({time.time()-t0:.0f}s). Extracting...', end=' ', flush=True)

    t1 = time.time()
    result = subprocess.run(['unzip', '-q', zip_local, '-d', LOCAL_DATA])
    os.remove(zip_local)

    if result.returncode != 0:
        print(f'FAILED (unzip error {result.returncode}) — zip may be corrupted on Drive.')
        failed.append(scene)
    else:
        print(f'done ({time.time()-t1:.0f}s).')

total = len(glob.glob(os.path.join(LOCAL_DATA, '*/exported/color/*.jpg')))
print(f'\nAll done. {total} frames in {LOCAL_DATA}')
if failed:
    print(f'\nFailed scenes (re-upload zips to Drive): {failed}')

## Cell 5 — Download Pretrained Weights
*(Skip on resume — weights persist for the session)*

In [7]:
WEIGHTS_PATH  = 'model/weights/indoor-lite-LA.ckpt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/final_proj/MatchFormer/model/weights/indoor-lite-LA.ckpt'

os.makedirs('model/weights', exist_ok=True)

if os.path.exists(WEIGHTS_PATH):
    print('Weights already present')
elif os.path.exists(DRIVE_WEIGHTS):
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_PATH)
    print('Copied weights from Drive')
else:
    print('WARNING: weights not found. Upload indoor-lite-LA.ckpt to Drive at:')
    print(' ', DRIVE_WEIGHTS)

## Cell 6 — Verify GPU & Sanity Check

Runs 50 steps on 5 pairs to confirm the pipeline works end-to-end.  
Loss should decrease. Takes ~1 min on L4.

In [8]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — no GPU!"}')
print(f'CUDA: {torch.cuda.is_available()}')

# # Sanity check: 5 pairs, 50 steps, reads from Drive (fine for 5 pairs)
# !python train_finetune.py \
#     --overfit \
#     --data_dir {DATA_ROOT} \
#     --checkpoint_dir {DRIVE_CKPT_DIR}/overfit \
#     --batch 4 \
#     --steps 50

# print('Sanity check done — loss should be decreasing!')

## Cell 7 — Full Fine-Tuning

**Auto-resume is on** — if Colab disconnects, re-run Cell 1 → 3 → 4 → this cell.  
It will pick up from the last checkpoint in `DRIVE_CKPT_DIR` automatically.

Key settings:
- `--data_dir` reads from **local SSD** (fast)
- `--precision bf16` — ~2x speedup on L4 via native bf16 tensor cores
- `--tau 50.0` — soft epipolar mask, better for multi-scene diversity
- `--save_every 500` — saves to Drive every 500 steps (40 checkpoints total)

In [ ]:
import wandb
wandb.login()

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python train_finetune.py \
    --data_dir {LOCAL_DATA} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 20000 \
    --tau 50.0 \
    --batch 8 \
    --workers 4 \
    --lr 1e-5 \
    --override_lr \
    --save_every 500 \
    --precision bf16 \
    --wandb \
    --wandb_project matchformer-finetune \
    --wandb_run phase7-lr1e5-focal-allneg-frozen

## Cell 8 — Benchmark After Training

In [9]:
import re, glob, shutil, os

LOCAL_DATA = '/content/scannet_data'
LOCAL_CKPT = 'model/weights/epipolar-finetuned.ckpt'

# Pick the highest epipolar-step=N.ckpt as the final weights
step_ckpts = [p for p in glob.glob(f'{DRIVE_CKPT_DIR}/epipolar-*.ckpt')
              if re.search(r'epipolar-step=(\d+)\.ckpt', p)]
if step_ckpts:
    TRAINED_CKPT = max(step_ckpts, key=lambda p: int(re.search(r'epipolar-step=(\d+)', p).group(1)))
else:
    TRAINED_CKPT = f'{DRIVE_CKPT_DIR}/last.ckpt'

print(f'Using checkpoint: {TRAINED_CKPT}')
os.makedirs('model/weights', exist_ok=True)
shutil.copy(TRAINED_CKPT, LOCAL_CKPT)
print(f'Copied to {LOCAL_CKPT}')

!python run_benchmark.py \
    --ckpt {LOCAL_CKPT} \
    --data_dir {LOCAL_DATA}/scene0000_00/exported \
    --num_pairs 100 \
    2>&1 | tee benchmark_epipolar.txt

shutil.copy('benchmark_epipolar.txt', f'{DRIVE_CKPT_DIR}/benchmark_epipolar.txt')
print('Results saved to Drive.')